In [1]:
import math
import multiprocessing
import os

from multiprocessing import Pool



# import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


import fnmatch
import sys
import itertools


#from sklearn.metrics import mean_squared_error, r2_score
#from sklearn.preprocessing import StandardScaler


from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

import cartopy.feature as cfeature
import dask


#import joblib

import xarray as xr
#from tqdm.notebook import tqdm



In [2]:
%run ./common_vars.py

%run ~/repos/spatialtuning/funs/plotting.py
%run ~/repos/spatialtuning/funs/utils.py


In [3]:
para = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/paras_v4_iteration1.csv", index_col=0)
ppe_pd = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index_col=0)
obs_pd = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index_col=0)

In [4]:
cli_nm = list(obs_dict.keys())
para_nm = list(para.columns)


In [5]:
para_norm = para.copy()
para_norm = (para_norm - para_norm.min())/(para_norm.max() - para_norm.min())


In [6]:
para_emu = xr.open_dataset("/glade/work/qingyuany/camml_v6/parameter_samples/para_samples.nc").to_dataframe()
para_emu

,micro_mg_max_nicons,micro_mg_vtrmi_factor,micro_mg_iaccr_factor,micro_mg_berg_eff_factor,micro_mg_accre_enhan_fact,micro_mg_homog_size,micro_mg_dcs,clubb_c1,clubb_C8,clubb_c11,...,zmconv_capelmt,seasalt_emis_scale,dust_emis_fact,sol_factb_interstitial,sol_factic_interstitial,microp_aero_wsub_min,microp_aero_wsubi_min,microp_aero_wsub_scale,microp_aero_wsubi_scale,microp_aero_npccn_scale
index,,,,,,,,,,,,,,,,,,,,,
0,0.596682,0.409964,0.438830,0.603460,0.037633,0.079303,0.403615,0.894232,0.315873,0.380472,...,0.722887,0.919643,0.350819,0.848872,0.831864,0.820495,0.045641,0.943457,0.687938,0.710679
1,0.131633,0.194753,0.969820,0.688295,0.344789,0.492622,0.254196,0.654934,0.504129,0.108424,...,0.648340,0.619922,0.315999,0.840754,0.002698,0.263238,0.516339,0.388326,0.584998,0.146721
2,0.924208,0.320887,0.873733,0.417682,0.636953,0.639374,0.426537,0.738028,0.841666,0.073009,...,0.561410,0.545908,0.673158,0.876488,0.703910,0.271816,0.867492,0.022760,0.774978,0.939132
3,0.933757,0.786729,0.888192,0.466601,0.098945,0.858185,0.792757,0.937193,0.779239,0.354424,...,0.237696,0.198019,0.542455,0.082092,0.776305,0.457705,0.480493,0.430490,0.966120,0.684267
4,0.165970,0.931272,0.210113,0.603715,0.314671,0.718420,0.113223,0.995542,0.944046,0.995805,...,0.106138,0.463397,0.461083,0.841091,0.379007,0.331414,0.992872,0.771853,0.076327,0.017796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0.753044,0.048855,0.842346,0.779287,0.226748,0.667534,0.750320,0.180039,0.501840,0.260636,...,0.593423,0.974859,0.037011,0.414848,0.742155,0.708742,0.905929,0.726438,0.361880,0.291495
999996,0.611086,0.163829,0.835600,0.479507,0.327686,0.680370,0.240949,0.766347,0.134996,0.284536,...,0.779102,0.022092,0.248538,0.363139,0.262933,0.063759,0.573833,0.696615,0.792158,0.096499
999997,0.292892,0.601946,0.336396,0.522423,0.558112,0.460850,0.653268,0.669575,0.855797,0.836616,...,0.657124,0.800076,0.940515,0.851387,0.465719,0.240731,0.529618,0.200661,0.225993,0.175015


In [ ]:
def gp_training_application(X, y, y_name, X_emu, path = "/glade/work/qingyuany/repo_data/spatialtuning/", no_sen = 2, no_restart = 10):
    y_norm = (y - y.mean())/y.sted()

    if ~(np.isnan(y_temp).all()):
        sage1d_temp = fit_all_gp_models_1d(y_temp.values, X.values, len1d = 0.4)
        sel_para_ind = sage1d_temp[:(no_sen * 2):2].astype(int)
        sel_para_ind = sel_para_ind.reshape(1,-1)

        kernel = C(1.0, (1e-3, 1)) * Matern(length_scale=0.001, nu=2.5, length_scale_bounds=(0.1, 3)) + WhiteKernel(noise_level=0.5, noise_level_bounds=(0.01, 0.9))
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer= no_restart, normalize_y=True)
        gp.fit(X.values[:,sel_para_ind], y_norm.values)
        
        joblib.dump(gp, path + "python_obj/" + y_name + "_gpmodel.pkl")

        y_mean_emu, y_std_emu = gp.predict(X_emu.values[:,sel_para_ind], return_std=True)

        pd.DataFrame(y_mean_emu, columns=[y_name]).to_csv(path + "emulated_ensemble" + "/gp_mean_" + zonal_var + ".csv")
        pd.DataFrame(y_std_emu, columns=[y_name]).to_csv(path + "emulated_ensemble" + "/gp_std_" + zonal_var + ".csv")

        return ([y_name, sel_para_ind, X.columns(sel_para_ind)])

In [23]:
sel_para_list = {}
generated_sample_path = "/glade/work/qingyuany/camml_v6/emulated_vars"


for zonal_var in tqdm(list(ppe_zonal_manual.columns)):
    y_temp = ppe_zonal_manual[zonal_var]
    y_temp = (y_temp - y_temp.mean())/(y_temp.std())

    if ~(np.isnan(y_temp).all()):
        sage1d_temp = fit_all_gp_models_1d(y_temp.values, para_v4_norm.values, len1d = 0.3)
        sel_paras_temp = sage1d_temp[:4:2].astype(int)
        print(sel_paras_temp)
        sel_para_list[zonal_var] = sel_paras_temp.reshape(1, -1)
    
        kernel = C(1.0, (1e-3, 1)) * Matern(length_scale=0.001, nu=0.9, length_scale_bounds=(0.01, 3)) + WhiteKernel(noise_level=0.5, noise_level_bounds=(0.001, 0.9))
    
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=8, normalize_y=True)
        gp.fit(para_v4_norm.values[:,sel_paras_temp], y_temp.values)
        
        joblib.dump(gp, '/glade/work/qingyuany/camml_v6/gps/' + zonal_var + "_gpmodel.pkl")

        y_mean_temp, y_std_temp = gp.predict(emu_para.values[:,sel_paras_temp], return_std=True)

        pd.DataFrame(y_mean_temp, columns=[zonal_var]).to_csv(generated_sample_path + "/gp_mean_" + zonal_var + ".csv")
        pd.DataFrame(y_std_temp, columns=[zonal_var]).to_csv(generated_sample_path + "/gp_std_" + zonal_var + ".csv")


meta_para = pd.DataFrame(np.concatenate(list(sel_para_list.values()), axis = 0), index = sel_para_list.keys(), columns = ["para_ind1", "para_ind2"])
meta_para.to_csv("/glade/work/qingyuany/camml_v6/gp_metadata_2para.csv", index = True)

  0%|          | 0/287 [00:00<?, ?it/s]

[32 30]


/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


[32 31]
[32 31]
[24  8]
[24  8]


/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


...

/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


[20 23]
[18 23]


/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


[16  8]


/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


'PRECT_-1_1_113_115'